# Functions in C

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Declare functions with prototypes, and explain why an argument changed inside a function stays the same outside (pass by value).
- Keep state between calls with a static variable, and split a problem into small functions.
- Write recursive functions, including functions that call each other.

**In this notebook**, one idea at a time (run each cell before moving on):
1. A function: type, name, parameters
2. Declare before use: prototypes
3. (void), and the empty parentheses trap
4. Arguments are copies
5. Local, global and static variables
6. Functions calling functions, and recursion
7. The standard library

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. A function: type, name, parameters

A C function looks like a Java `static` method with no class around it: a **return type**, a name, **typed parameters** in parentheses, and a body. `return` gives back the result. A function that returns nothing has the type `void`.

In [ ]:
#include <stdio.h>

int square(int x) {
    return x * x;
}

void print_line(int width) {
    for (int i = 0; i < width; i++) {
        printf("-");
    }
    printf("\n");
}

int main(void) {
    print_line(12);
    printf("5 squared is %d\n", square(5));
    printf("square(square(3)) is %d\n", square(square(3)));
    print_line(12);
    return 0;
}

🧪 **Try it:** write `int cube(int x)` using `square`, and print `cube(4)`.

## 2. Declare before use: prototypes

The compiler reads the file **from top to bottom**. When it reaches a call, it must already know the function: its return type and parameters. Either define the function above the call, or put a **prototype** at the top, the function's first line followed by `;`, and define it anywhere below. Prototypes let `main` stay at the top, where it's easy to find.

In [ ]:
#include <stdio.h>

double average(int a, int b);      // prototype: "this function exists"

int main(void) {
    printf("%.1f\n", average(7, 8));
    return 0;
}

double average(int a, int b) {     // the definition, below
    return (a + b) / 2.0;
}

🧪 **Try it:** delete the prototype line and read the error: the compiler met `average` before knowing it.

## 3. (void), and the empty parentheses trap

For a function with no parameters, write `(void)`: `int next_ticket(void)`. In C17, an empty `()` in a declaration means "parameters not stated", so the compiler can't check that calls pass the right arguments. Likewise, a function that returns a value must `return` on every path: falling off the end gives the caller garbage (and a warning).

In [ ]:
#include <stdio.h>

int lucky(void) {
    return 7;
}

const char *sign(int n) {
    if (n < 0) {
        return "negative";
    }
    if (n > 0) {
        return "positive";
    }
    return "zero";
}

int main(void) {
    printf("%d %s %s\n", lucky(), sign(-4), sign(0));
    return 0;
}

🧪 **Try it:** call `lucky(5)` and read the error: `(void)` lets the compiler catch it. Then delete `return "zero";` and read the warning.

## 4. Arguments are copies

When you call a function, each argument's **value is copied** into the parameter. The function works on its copy, and the caller's variable doesn't change. To give a result back, `return` it. (To change the caller's variable itself, C uses pointers, in a later unit.) The slides show the copy on the call stack.

#### 🎞️ Arguments are copies

![add_one(n) gets its own copy x of n; changing x leaves main's n at 5](copy-call-steps.svg)

In [ ]:
#include <stdio.h>

void add_one(int x) {
    x++;
    printf("inside add_one: x = %d\n", x);
}

int plus_one(int x) {
    return x + 1;
}

int main(void) {
    int n = 5;
    add_one(n);
    printf("after add_one: n = %d\n", n);
    n = plus_one(n);
    printf("after n = plus_one(n): n = %d\n", n);
    return 0;
}

🧪 **Try it:** predict the three lines before running. Then rename `x` to `n` in `add_one`: does it change anything? (No: it's still a separate copy.)

### ✏️ Exercise: The smallest of three

Return the smallest of three ints. Declare a helper `min2` and call it twice.

In [ ]:
int min2(int a, int b) {
    return a < b ? a : b;
}

int min3(int a, int b, int c) {
    // TODO: return min2(min2(a, b), c)
    return a;
}

int main(void) {
    check_int("first", min3(1, 5, 9), 1);
    check_int("middle", min3(5, -2, 9), -2);
    check_int("last", min3(5, 4, 3), 3);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int min2(int a, int b) {
    return a < b ? a : b;
}

int min3(int a, int b, int c) {
    return min2(min2(a, b), c);
}

int main(void) {
    check_int("first", min3(1, 5, 9), 1);
    check_int("middle", min3(5, -2, 9), -2);
    check_int("last", min3(5, 4, 3), 3);
    return 0;
}
```

</details>

## 5. Local, global and static variables

A variable created inside a function is **local**: it's created on each call and disappears when the function returns. A variable outside every function is **global**: any function can change it, which makes bugs hard to track, so use them sparingly. A **`static` local** is in between: only its function sees it, but it's created once and **keeps its value between calls**.

In [ ]:
#include <stdio.h>

int calls_local(void) {
    int count = 0;           // created again on every call
    count++;
    return count;
}

int calls_static(void) {
    static int count = 0;    // created once, keeps its value
    count++;
    return count;
}

int main(void) {
    for (int i = 0; i < 3; i++) {
        printf("local: %d, static: %d\n", calls_local(), calls_static());
    }
    return 0;
}

🧪 **Try it:** try to print `count` inside `main`: neither one is visible there.

### ✏️ Exercise: A counter that remembers

Each call returns the next id: 1, then 2, then 3. A `static` local keeps its value between calls.

In [ ]:
int next_id(void) {
    // TODO: keep a static counter and increase it
    return 0;
}

int main(void) {
    check_int("first call", next_id(), 1);
    check_int("second call", next_id(), 2);
    check_int("third call", next_id(), 3);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
int next_id(void) {
    static int last = 0;
    last++;
    return last;
}

int main(void) {
    check_int("first call", next_id(), 1);
    check_int("second call", next_id(), 2);
    check_int("third call", next_id(), 3);
    return 0;
}
```

</details>

## 6. Functions calling functions, and recursion

Functions call other functions, and themselves, just like Java methods. Each call gets its own frame on the call stack, with its own copies of the parameters and local variables.

In [ ]:
#include <stdio.h>

int gcd(int a, int b) {
    if (b == 0) {
        return a;               // base case
    }
    return gcd(b, a % b);
}

int lcm(int a, int b) {
    return a / gcd(a, b) * b;
}

int main(void) {
    printf("gcd(12, 18) = %d\n", gcd(12, 18));
    printf("lcm(4, 6) = %d\n", lcm(4, 6));
    return 0;
}

🧪 **Try it:** trace `gcd(12, 18)` by hand: gcd(18, 12), gcd(12, 6), gcd(6, 0)… Then add a `printf` at the start of `gcd` to check.

## 7. The standard library

C comes with a standard library, split into headers you `#include`: `<stdio.h>` (input and output), `<stdlib.h>` (`abs`, `atoi`, `malloc`…), `<string.h>` (text), `<ctype.h>` (`isdigit`, `toupper`…), `<stdint.h>` (fixed-size integers). The header holds the prototypes, so the compiler can check your calls.

In [ ]:
#include <ctype.h>
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    printf("%d\n", abs(-12));
    printf("%d\n", atoi("345") + 1);
    printf("%c %d\n", toupper('q'), isdigit('7') != 0);
    return 0;
}

🧪 **Try it:** remove `#include <ctype.h>` and read the error about `toupper`: without the header, the compiler doesn't know the function.

## In short

A C function has a return type, a name and typed parameters, like a Java `static` method with no class around it. C reads the file from top to bottom, so a function must be **declared** before it is used: either defined above, or announced by a **prototype**.

```c
#include <stdio.h>

int next_ticket(void);           // prototype: defined below

int main(void) {
    for (int i = 0; i < 3; i++) {
        printf("Ticket %d\n", next_ticket());   // 1, 2, 3
    }
    return 0;
}

int next_ticket(void) {
    static int last = 0;          // created once, keeps its value between calls
    last++;
    return last;
}
```

Arguments are **copied** into the parameters: a function that changes its parameter changes only its copy. To give back a result, `return` it.

**Watch out:** write `(void)` for "no parameters". An empty `()` in a declaration means "parameters not stated" in C17, so the compiler can't check the call.

## Readings

- [Functions](https://beej.us/guide/bgc/html/split/functions.html) (Beej's Guide to C Programming, chapter 4, 8 min): Passing by value, function prototypes, and why int f(void) is not the same as int f() in older C.
- [Static variables](https://www.freecodecamp.org/news/the-c-beginners-handbook/#heading-static-variables) (freeCodeCamp, The C Beginner's Handbook, 3 min): Optional: a static local variable keeps its value from one call to the next. The "Global variables" section right after shows the other way to share state, and why it is riskier.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Dice Roller** · Easy
- **Luhn Check** · Easy
- **Month Calendar** · Medium
- **Change Making** · Medium
- **Expression Evaluator** · Hard